# Sesión 12 · Automatización con n8n y cierre del Módulo 1

**AI Engineer I: Fundamentos de IA Generativa** · Módulo 1 · Edutek

---

## 🎯 Objetivos

Al terminar este notebook podrás:

1. **Decidir** cuándo conviene un workflow de bajo código (n8n) y cuándo código propio, y qué se gana y se pierde.
2. **Leer y explicar un workflow de n8n**: triggers, nodos, *items*, expresiones, credenciales, ejecuciones y depuración.
3. **Armar un agente en n8n** (AI Agent con modelo, memoria y herramientas), pedir **salida estructurada** y usar **sub-workflows** como herramientas.
4. **Conectar MCP**: n8n como cliente (*MCP Client Tool*) y como servidor (*MCP Server Trigger*).
5. **Diseñar un flujo con aprobación humana**: webhook autenticado, clasificación estructurada, enrutamiento por severidad, agente que *propone*, aprobación antes de actuar, reintentos y *error workflow*.
6. **Nombrar los límites reales** de n8n: versionado, pruebas, licencia *fair-code*, operación en *self-hosting* y seguridad.
7. **Repasar el Módulo 1** completo y saber qué viene en el Módulo 2.

> Esta sesión es **teoría con demostración**: el notebook explica los conceptos y trae unas pocas celdas de Python (un mini-motor de *items*, un simulador de reintentos, un revisor de workflows y, **opcionalmente**, llamadas HTTP al n8n local). Los workflows viven en `ai-agents-playground/n8n-workflows`.

## 🗺️ Mapa

| Parte | Tema | ¿Consume dinero? |
| --- | --- | --- |
| 0 | Preparar el entorno | No |
| 1 | Bajo código vs código propio | No |
| 2 | Anatomía de un workflow | No |
| 3 | Leer un workflow exportado (y revisarlo con código) | No |
| 4 | Agentes en n8n | No |
| 5 | MCP en n8n | No |
| 6 | El flujo del curso: triaje de tickets con aprobación humana | No |
| 7 | Errores, reintentos y sub-workflows | No |
| 8 | Límites reales y operación | No |
| 9 | (Opcional) Llamar al n8n local desde Python | No en modo `simulado`; con modelo real, centavos |
| 10 | La demostración | No en modo `simulado` |
| 11 | Cierre de la sesión y del Módulo 1 | — |

> 📌 **Versión.** Todo se probó con **n8n 2.47.7** (imagen `n8nio/n8n:2.47.7`, noviembre de 2026). n8n publica una versión menor casi cada semana y ya hay una serie 3.x en beta: **fija la versión** en tu `docker-compose.yml` y lee las notas de versión antes de actualizar. Confirma los detalles en [docs.n8n.io](https://docs.n8n.io).

---
# Parte 0 · Preparar el entorno

In [ ]:
import sys

print("Python:", sys.version.split()[0])
assert sys.version_info[:2] == (3, 12), (
    "Esta sesión está fijada a Python 3.12. Ejecuta `uv sync` dentro de la carpeta "
    "s12-n8n-automatizacion y selecciona el kernel .venv (ver README)."
)

In [ ]:
import json
import os
import time
from pathlib import Path

import requests

# Interruptores. Todo el notebook corre sin n8n; solo la Parte 9 necesita el n8n del playground levantado.
N8N_LOCAL = False                   # ← True si tienes n8n en http://localhost:5678 con los workflows importados (Parte 9)
N8N_URL = os.getenv("N8N_URL", "http://localhost:5678")

# Dónde están los workflows exportados (repositorio ai-agents-playground, carpeta n8n-workflows).
# Si lo clonaste en otro sitio, define la variable de entorno N8N_WORKFLOWS_DIR.
CANDIDATOS = [
    Path(os.getenv("N8N_WORKFLOWS_DIR", "")),
    Path("../../../ai-agents-playground/n8n-workflows/workflows"),
    Path("../../ai-agents-playground/n8n-workflows/workflows"),
    Path("ai-agents-playground/n8n-workflows/workflows"),
]
DIR_WORKFLOWS = next((p for p in CANDIDATOS if str(p) not in ("", ".") and p.is_dir()), None)
print("Workflows exportados:", DIR_WORKFLOWS.resolve() if DIR_WORKFLOWS else "no encontrados (las Partes 3 y 7 usarán un ejemplo mínimo)")

---
# Parte 1 · Bajo código vs código propio

## Qué es n8n

**n8n** es una herramienta de **automatización de flujos de trabajo** (*workflow automation*) con editor visual: dibujas nodos conectados y cada nodo hace una cosa (recibir un webhook, llamar a una API, decidir, llamar a un modelo). Se puede **auto-hospedar** (Docker, Kubernetes) o usar su nube. Trae cientos de integraciones y nodos de IA (agentes, memoria, herramientas, MCP) construidos sobre **LangChain**.

**Analogía para infraestructura:** es a la integración de sistemas lo que un **pipeline de CI/CD** o un **playbook de Ansible** son a la operación: una definición declarativa (aquí, un JSON) de pasos con disparadores, condiciones y manejo de errores; con editor gráfico, historial de ejecuciones y un runtime que ya resuelve lo repetitivo (autenticación, reintentos, programación).

## ¿Cuándo bajo código y cuándo código propio?

| Criterio | Workflow de bajo código (n8n) | Código propio (LangChain, LangGraph, etc.) |
| --- | --- | --- |
| **Integrar muchos servicios** (CRM, correo, chat, hojas) | ✅ Nodos listos, autenticación resuelta | Hay que escribir y mantener cada cliente |
| **Velocidad del primer prototipo** | ✅ Horas | Días |
| **Quién lo mantiene** | También perfiles no programadores | Solo quien programa |
| **Lógica compleja, algoritmos, estructuras de datos** | ⚠️ Se vuelve un "lienzo de espagueti" | ✅ Lenguaje completo |
| **Pruebas automáticas y revisión de cambios** | ⚠️ Limitado (JSON grandes, pocas pruebas nativas) | ✅ pytest, CI, *code review* |
| **Control fino de un agente** (estado, grafos, reanudación) | ⚠️ El nodo de agente es una caja | ✅ LangGraph |
| **Costo de operar** | Un servicio más que mantener (base, actualizaciones, seguridad) | Lo que ya operas |
| **Observabilidad y auditoría** | ✅ Historial de ejecuciones listo | La construyes |

**Regla práctica:** n8n es excelente como **pegamento y orquestación de procesos** (disparadores, integraciones, aprobaciones, notificaciones). Cuando el corazón del problema es **lógica o razonamiento complejo**, escríbelo como código (un servicio, un agente en LangGraph) y deja que n8n lo **llame** por HTTP o por MCP. No son excluyentes: el workflow de esta sesión mezcla nodos con pequeños fragmentos de código y llama a un agente.

> 💬 **Para pensar:** en tu empresa, ¿qué proceso hoy está hecho con scripts sueltos y cron, y se beneficiaría de tener disparadores, reintentos e historial de ejecuciones listos? ¿Y cuál **no** debería moverse a un lienzo visual?

## Workflow fijo vs agente (repaso del Módulo 1)

Un **workflow** tiene un camino que **tú** dibujas. Un **agente** decide el camino con un modelo. En la práctica se **combinan**: el workflow da la estructura segura (validar, enrutar, aprobar, auditar) y el agente se usa en un **paso acotado** (diagnosticar, redactar). Es la misma idea de las sesiones 8 a 11: empieza por lo más simple que cumpla el objetivo, y agrega autonomía solo donde haga falta.

---
# Parte 2 · Anatomía de un workflow

## Las piezas

| Pieza | Qué es |
| --- | --- |
| **Trigger** (disparador) | El nodo que inicia el workflow: *Webhook*, *Chat Trigger*, *Schedule*, *Error Trigger*, *MCP Server Trigger*, "cuando me llaman" (sub-workflow)… |
| **Nodo** | Un paso: acción (HTTP Request, Code, Set), lógica (IF, Switch, Merge, Wait) o IA (AI Agent, Information Extractor, modelos, memoria, herramientas) |
| **Conexión** | Línea que lleva los datos de la salida de un nodo a la entrada de otro. Los nodos de IA tienen conexiones **especiales** (modelo, memoria, herramienta, parser) |
| **Item** | La unidad de datos. Un nodo recibe una **lista de items**, cada uno `{"json": {...}}` (y opcionalmente binarios), y **procesa cada uno** |
| **Expresión** | Un valor calculado entre `{{ }}`, por ejemplo `{{ $json.ticket.asunto }}` o `{{ $('Validar entrada').first().json.modo }}` |
| **Credencial** | Secreto guardado **cifrado** en n8n y referenciado por nombre desde los nodos; no viaja dentro del workflow |
| **Ejecución** | Una corrida del workflow con **todos los datos de entrada y salida de cada nodo**, guardada para depurar |

## Items y expresiones

- Cada nodo recibe **todos los items** de su entrada y entrega una lista nueva. Un nodo como *Set* o *HTTP Request* se aplica **a cada item**; uno como *Code* puede procesarlos todos a la vez.
- `{{ $json.campo }}` lee el item **actual** de la entrada del nodo. `{{ $('Nombre del nodo').first().json.campo }}` lee la salida de **otro nodo** (por eso los nombres de los nodos importan y conviene que sean claros).
- Las expresiones usan JavaScript. Se activan con `=` al inicio del valor del campo (en el JSON exportado verás `"={{ ... }}"`).
- `$execution.id`, `$execution.resumeFormUrl` y otras variables dan datos de la ejecución.

### Experimento: un mini-motor de items (sin n8n)

Para fijar la idea, este mini-motor reproduce lo esencial: una lista de nodos, cada uno una función que recibe **items** y devuelve **items**, con acceso a la salida de nodos anteriores. No es n8n, es la **forma** de su modelo de datos.

In [ ]:
from typing import Callable

Item = dict            # {"json": {...}}
Nodo = Callable[[list[Item], dict], list[Item]]


def correr(nodos: list[tuple[str, Nodo]], items: list[Item]) -> dict[str, list[Item]]:
    # Ejecuta los nodos en orden y guarda la salida de cada uno (como el historial de una ejecución).
    salidas: dict[str, list[Item]] = {}
    for nombre, fn in nodos:
        items = fn(items, salidas)
        salidas[nombre] = items
    return salidas


def validar(items, previos):                      # ≈ nodo Code
    out = []
    for it in items:
        b = it["json"].get("body", {})
        errores = [] if len(b.get("descripcion", "")) >= 10 else ["descripcion muy corta"]
        out.append({"json": {"valido": not errores, "errores": errores, "ticket": b}})
    return out


def solo_validos(items, previos):                 # ≈ nodo IF (rama verdadera)
    return [it for it in items if it["json"]["valido"]]


def clasificar(items, previos):                   # ≈ nodo Set / Code con reglas
    out = []
    for it in items:
        txt = it["json"]["ticket"]["descripcion"].lower()
        severidad = "alta" if "no responde" in txt else "baja"
        out.append({"json": {**it["json"], "severidad": severidad}})     # un item nuevo: no se modifica el de la entrada
    return out


webhook = [{"json": {"body": {"descripcion": "El servidor de archivos no responde"}}},
           {"json": {"body": {"descripcion": "corto"}}}]
ejecucion = correr([("Validar", validar), ("¿Válido?", solo_validos), ("Clasificar", clasificar)], webhook)

for nombre, salida in ejecucion.items():
    print(f"{nombre:<12} → {len(salida)} item(s)", [s["json"].get("severidad", s["json"].get("valido")) for s in salida])
print("\n'Expresión' $('Validar').first().json.ticket →", ejecucion["Validar"][0]["json"]["ticket"])

**Qué observar.** Entraron **2 items** y la rama "válido" dejó pasar **1**: cada nodo trabaja sobre listas. Y `ejecucion["Validar"]` guarda la salida de cada paso: es exactamente lo que n8n **guarda por ejecución** y por lo que puedes abrir una corrida pasada y ver qué datos tenía cada nodo.

## Credenciales

- Se crean una vez (*Credentials*) y se **referencian** desde los nodos. En el JSON exportado solo queda `{"id": ..., "name": ...}`: **el secreto no sale del servidor**.
- Se guardan **cifradas** en la base con la clave `N8N_ENCRYPTION_KEY`. Si pierdes esa clave, las credenciales guardadas no se pueden leer.
- Tipos útiles hoy: *Header Auth* (para proteger un webhook o llamar a una API), *OpenAI*, *Anthropic*, OAuth2 para servicios en la nube.

## Ejecuciones y depuración

| Herramienta | Para qué |
| --- | --- |
| **Ejecutar paso / ejecutar workflow** en el editor | Ver datos de entrada y salida de cada nodo |
| **Pin data** (fijar datos) | Congelar la salida de un nodo (por ejemplo, un webhook) para iterar sin volver a dispararlo |
| **Lista de ejecuciones** | Revisar corridas de producción (exitosas y fallidas) con sus datos |
| **Depurar en el editor** | Cargar los datos de una ejecución fallida en el editor y corregir |
| **Notas y nombres claros** | El mejor "comentario" de un workflow |
| **URL de prueba vs de producción** de un webhook | La de prueba solo escucha mientras estás probando en el editor; la de producción requiere que el workflow esté **publicado/activo** |

> ⚠️ **Los datos de ejecución son datos sensibles.** Guardan lo que recibió y devolvió cada nodo, incluido el texto de los tickets. Define cuánto tiempo se conservan (el compose del laboratorio borra a los 7 días) y quién puede verlas.

---
# Parte 3 · Leer un workflow exportado (y revisarlo con código)

Un workflow de n8n es un **archivo JSON**: lista de `nodes` (con su `type`, `parameters` y `credentials`) y un mapa de `connections`. Eso permite **versionarlo en Git, compararlo y revisarlo con código**. Esta celda carga el triaje del playground y resume su estructura.

In [ ]:
def cargar(nombre: str) -> dict | None:
    if DIR_WORKFLOWS is None:
        return None
    ruta = DIR_WORKFLOWS / nombre
    return json.loads(ruta.read_text(encoding="utf-8")) if ruta.exists() else None


triaje = cargar("02-triaje-tickets.json")
if triaje is None:
    print("⏭️ No se encontró el playground; define N8N_WORKFLOWS_DIR (ver Parte 0).")
else:
    from collections import Counter

    tipos = Counter(n["type"].split(".")[-1] for n in triaje["nodes"] if "stickyNote" not in n["type"])
    print(f"{triaje['name']}\n{len(triaje['nodes'])} nodos (incluye notas adhesivas)\n")
    for tipo, cuantos in tipos.most_common():
        print(f"  {cuantos:>2} × {tipo}")
    ia = sorted({k for s in triaje["connections"].values() for k in s if k.startswith("ai_")})
    print("\nConexiones especiales de IA:", ia)
    print("Error workflow:", triaje["settings"].get("errorWorkflow"))

**Qué leer.** Hay nodos de lógica (`if`, `switch`, `code`, `wait`), de red (`httpRequest`, `webhook`, `respondToWebhook`), sub-workflows (`executeWorkflow`) y de IA (`agent`, `informationExtractor`, modelos, herramientas). Las conexiones `ai_languageModel`, `ai_tool` y `ai_outputParser` son las "tomas" especiales de los nodos de IA. Y `errorWorkflow` apunta al workflow que avisa cuando algo falla.

## Ejercicio: un revisor automático de workflows

Como son JSON, puedes escribir **reglas de revisión** (un *linter*) que corran en CI antes de importar. Esta función revisa cuatro cosas: webhooks sin autenticación, secretos embebidos, llamadas HTTP sin reintentos y uso de variables de entorno en nodos Code.

In [ ]:
def revisar(wf: dict) -> list[str]:
    hallazgos = []
    for n in wf["nodes"]:
        p, nombre, tipo = n.get("parameters", {}), n["name"], n["type"].split(".")[-1]
        if tipo == "webhook" and p.get("authentication", "none") == "none":
            hallazgos.append(f"🔓 {nombre}: webhook SIN autenticación")
        for cred in n.get("credentials", {}).values():
            if set(cred) - {"id", "name"}:
                hallazgos.append(f"🔑 {nombre}: la credencial lleva más que id y nombre (¿secreto embebido?)")
        if tipo == "httpRequest" and not n.get("retryOnFail"):
            hallazgos.append(f"🔁 {nombre}: llamada HTTP sin reintentos")
        if tipo == "code" and "$env" in p.get("jsCode", ""):
            hallazgos.append(f"⚠️ {nombre}: lee variables de entorno desde un nodo Code")
    return hallazgos


if triaje:
    print("Triaje del curso →", revisar(triaje) or "sin hallazgos ✅")

malo = {"nodes": [
    {"name": "Webhook abierto", "type": "n8n-nodes-base.webhook", "parameters": {"path": "x"}},
    {"name": "Llamar API", "type": "n8n-nodes-base.httpRequest", "parameters": {}, "credentials": {"httpHeaderAuth": {"id": "1", "name": "k", "data": {"value": "secreto"}}}},
    {"name": "Código", "type": "n8n-nodes-base.code", "parameters": {"jsCode": "return [{json:{k:$env.API_KEY}}]"}},
]}
print("\nWorkflow de ejemplo mal hecho:")
for h in revisar(malo):
    print(" ", h)

> 💬 **Para pensar:** ¿qué otras reglas añadirías (por ejemplo, "todo workflow con un nodo de escritura debe tener un nodo *Wait* antes", o "toda herramienta del agente debe tener descripción")? El playground trae pruebas `pytest` de este estilo (`tests/test_workflows.py`).

---
# Parte 4 · Agentes en n8n

## El nodo AI Agent

El nodo **AI Agent** es un agente con **bucle de herramientas** (el mismo concepto de las sesiones 8 y 9), configurado con **conexiones** en lugar de código:

| Toma del nodo | Qué se conecta | Equivalente en LangChain |
| --- | --- | --- |
| **Chat Model** | OpenAI, Anthropic, Ollama, Azure…; con un servidor compatible con OpenAI (LM Studio) basta cambiar la URL de la credencial | El `model` de `create_agent` |
| **Memory** | *Simple Memory* (ventana de N mensajes en RAM, por `sessionId`) o una memoria persistente (Postgres, Redis…) | El *checkpointer* y el `thread_id` |
| **Tool** (varias) | *Code Tool*, *HTTP Request Tool*, otro workflow como herramienta (*Call n8n Workflow Tool*), *MCP Client Tool*, integraciones | Las herramientas `@tool` |
| **Output Parser** | *Structured Output Parser* (esquema JSON) | `response_format` |
| **Opciones** | Mensaje de sistema, **máximo de iteraciones** (tope del bucle) | `system_prompt` y límites del bucle |

**Lo que importa (y es igual que en código):**

- El modelo **pide** herramientas; n8n las **ejecuta** y devuelve el resultado al modelo.
- Lo único que el modelo sabe de una herramienta es su **nombre y su descripción** (y el esquema de entradas). En n8n el nombre que ve el modelo sale del **nombre del nodo** de la herramienta: llámalo `buscar_runbook`, no "Code Tool 3".
- Un agente **cuesta varias llamadas** por mensaje. Pon un máximo de iteraciones y un modelo económico.
- La **memoria simple vive en el proceso de n8n**: se pierde al reiniciar y no escala a varias instancias. Para producción, una memoria persistente.

## Salida estructurada

Dos formas, según el caso:

| Nodo | Cuándo |
| --- | --- |
| **Information Extractor** (con esquema) | Sacar campos de un texto (clasificar un ticket, extraer entidades): una llamada, sin herramientas |
| **Structured Output Parser** conectado al AI Agent | Que la **respuesta final del agente** tenga forma fija |

> ⚠️ **Igual que en la sesión 9: el esquema garantiza la forma, no la verdad.** Por eso el flujo del curso **vuelve a validar en código** (nodos *Code*) que la severidad esté en la lista, que la acción esté en la lista blanca y que la confianza sea un número. No dependas de que el modelo se porte bien.

## Sub-workflows como herramientas

El nodo **Call n8n Workflow Tool** convierte **otro workflow** en una herramienta del agente: su disparador es "cuando me llaman" (*Execute Workflow Trigger*) y sus entradas son los parámetros que el modelo rellena. Ventajas: la herramienta se prueba y versiona **por separado**, se reutiliza (en el chat, en el triaje, en el servidor MCP) y puede tener sus propios reintentos y permisos.

## Agente vs. workflow: dónde poner la decisión

| Pregunta | Si la respuesta es **sí** |
| --- | --- |
| ¿El camino es siempre el mismo? | Workflow (nodos), sin agente |
| ¿Hay que **interpretar texto libre** o elegir entre herramientas según el caso? | Agente en **ese paso** |
| ¿El paso **modifica** algo? | Que lo ejecute el workflow **después de una aprobación**, no el agente |
| ¿Necesitas control fino del estado, ciclos o varios agentes? | Código propio (LangGraph) y n8n lo llama |

En el triaje de esta clase el agente **solo diagnostica y propone**: no tiene herramientas de escritura. La acción la ejecuta el flujo, después de que una persona la aprueba.

---
# Parte 5 · MCP en n8n

En la sesión 10 viste el **Model Context Protocol**: un estándar para exponer herramientas a cualquier cliente. n8n puede estar en **los dos lados**:

| Nodo | Rol | Qué hace |
| --- | --- | --- |
| **MCP Client Tool** | Cliente | Se conecta a un servidor MCP (por HTTP *streamable* o SSE) y **ofrece sus herramientas al AI Agent**. Se puede incluir todas, solo algunas o todas menos algunas |
| **MCP Server Trigger** | Servidor | Expone como herramientas MCP **los nodos de herramienta conectados a él**, en una URL (`/mcp/<ruta>`). Admite autenticación por cabecera o Bearer |

```text
 Cliente MCP (otro agente, IDE, otro n8n)            n8n
 ──────────────────────────────────────  HTTP  ┌───────────────────────────┐
  initialize · tools/list · tools/call  ─────► │ MCP Server Trigger        │
                                               │   ├─ Code Tool            │
                                               │   └─ Call n8n Workflow    │
                                               └───────────────────────────┘
```

**En el laboratorio:**

- El workflow `07-servidor-mcp` expone `estado_servicio` y `buscar_runbook` con autenticación por cabecera. Se probó con `curl` (`initialize`, `tools/list`).
- Los workflows de agente traen un **MCP Client Tool desactivado** que apunta a ese servidor. Conectarlo es **opcional**; y también lo es apuntarlo al **servidor MCP de la Sesión 10** (HubSpot) por HTTP: cambia la URL a `http://host.docker.internal:<puerto>/mcp` y la autenticación.

**Lo de seguridad de la Sesión 10 sigue valiendo:** un servidor MCP es una **superficie de ataque**. Autenticación obligatoria, herramientas de mínimo privilegio, descripciones sin instrucciones ocultas, y **aprobación humana** para escrituras. En n8n, el servidor MCP corre con los permisos de las credenciales de sus herramientas: cuidado con qué nodos conectas.

> 💬 **Para pensar:** ¿qué ganas exponiendo tus workflows como un servidor MCP (reutilizar procesos de la empresa desde cualquier agente)? ¿Qué riesgo agregas (cualquier cliente que logre autenticarse puede disparar esos procesos)?

---
# Parte 6 · El flujo del curso: triaje de tickets con aprobación humana

Un flujo "no tan básico", pensado para **demostrarse en 15 minutos** y reproducirse en casa **sin llaves** (modo `simulado`) o con un modelo (`llm`).

## Qué hace

1. Un **webhook autenticado** recibe un ticket de TI (JSON ficticio).
2. Valida el formato y los tamaños, y **responde `202` enseguida**; el resto sigue en segundo plano.
3. **Clasifica y extrae** datos con un LLM con salida estructurada (en `simulado`, con reglas de palabras clave).
4. Un **sub-workflow** consulta la CMDB (llamada HTTP con **reintentos**) y devuelve el activo afectado.
5. Una **regla determinista** sube la severidad si el activo es crítico y está en producción.
6. Un **Switch** enruta por severidad: crítica y alta van al **agente**; media y baja reciben una respuesta estándar.
7. El **agente** (con herramientas de estado y de runbooks) **propone** una acción. Una validación en código la obliga a estar en una **lista blanca**.
8. **Aprobación humana**: n8n pausa (*Wait*) y espera un formulario (Aprobar o Rechazar). Si nadie responde en 30 minutos, **caduca** y no se ejecuta nada.
9. Si se aprueba, se ejecuta la acción en **`dry_run`** (simulada); se **notifica** y se **registra la auditoría**.
10. Si algo falla, un **error workflow** avisa a la guardia.

```mermaid
flowchart LR
    W[Webhook<br/>autenticado] --> V[Validar] --> R202[202]
    R202 --> C[Clasificar<br/>estructurado] --> E[Sub-workflow<br/>CMDB] --> P[Prioridad]
    P --> S{Switch}
    S -->|crítica/alta| A[Agente<br/>propone] --> L[Lista blanca] --> H[/Aprobación humana/]
    H -->|aprobar| X[Acción dry-run]
    H -->|rechazar/caduca| F
    X --> F[Notificar + auditoría]
    S -->|media/baja| Q[Respuesta estándar] --> F
    ERR[(Error workflow)] -.-> G[Aviso a guardia]
```

## Decisiones de diseño (y por qué)

| Decisión | Por qué |
| --- | --- |
| El agente **no tiene herramientas de escritura** | Un modelo puede equivocarse o ser engañado por el texto del ticket; la acción pasa por una persona |
| **Lista blanca** de acciones en código y en el esquema | El esquema da la forma; el código impone el contenido permitido |
| Reglas de negocio **en nodos**, no en el prompt (subir severidad por activo crítico) | Lo determinista no debe depender de un modelo |
| `dry_run` por defecto y caducidad de la aprobación | Escrituras reversibles, acotadas y sin aprobaciones eternas |
| Modo `simulado` y servicios simulados dentro de n8n | Se puede enseñar, probar y repetir sin cuentas ni costo |
| El texto del ticket se trata como **dato** (`05_inyeccion`) | La **inyección indirecta** (sesión 9) también aplica aquí |
| Responder `202` temprano y trabajar después | El cliente no espera a un agente ni a una persona |

## Los tickets de ejemplo

| Archivo | Ruta del flujo |
| --- | --- |
| `01_critico_vpn` | Crítica → agente → aprobación → acción en `dry_run` |
| `02_alto_correo` | Alta → agente → aprobación (se rechaza) |
| `03_media_wiki_lenta` / `04_bajo_contrasena` | Media o baja → respuesta estándar |
| `05_inyeccion` | El texto intenta dar órdenes al asistente |
| `06_falla_cmdb` | La CMDB falla → reintentos → error workflow |
| `07_invalido` | Validación → `400` |

Todos los datos son **ficticios**.

---
# Parte 7 · Errores, reintentos y sub-workflows

## Cuatro niveles de manejo de errores

| Nivel | Qué hace | Dónde |
| --- | --- | --- |
| **Reintento del nodo** (*Retry On Fail*) | Vuelve a ejecutar el nodo si falla (número de intentos y espera entre ellos) | Ajustes del nodo (útil para fallas **transitorias**: red, 503) |
| **Qué hacer ante el error** (*On Error*) | *Detener el workflow* (por defecto), *continuar* con el último dato válido, o *continuar usando la salida de error* (una rama aparte para manejarlo) | Ajustes del nodo |
| **Error workflow** | Un workflow que empieza con **Error Trigger** y corre cuando **otro workflow falla**; recibe el nombre del workflow, el nodo, el mensaje y el enlace a la ejecución | *Settings → Error workflow* del workflow vigilado |
| **Caducidad de esperas** | El nodo *Wait* puede limitar el tiempo de espera; al vencer, la ejecución sigue por la rama de "expirada" | Opciones del nodo *Wait* |

**Cuándo cuál.** Un 503 de una API: reintento. Un dato que puede faltar sin romper el flujo: *continuar*. Una falla de la que alguien debe enterarse: error workflow. Una persona que no contesta: caducidad.

> ⚠️ **Los reintentos repiten efectos.** Reintentar una llamada que **escribe** puede duplicar la acción. Reintenta lecturas; en escrituras exige **idempotencia** (una llave de operación) o no reintentes.

### Experimento: reintentos

Esta celda simula una API que falla dos veces y luego responde, con tres intentos, como el nodo *HTTP Request* con *Retry On Fail*.

In [ ]:
def api_inestable(intentos_hasta_exito: int):
    llamadas = {"n": 0}

    def llamar():
        llamadas["n"] += 1
        if llamadas["n"] < intentos_hasta_exito:
            raise ConnectionError("503 Service Unavailable")
        return {"host": "vpn-gw-01", "criticidad": "alta"}

    return llamar


def con_reintentos(fn, max_tries=3, espera_ms=1000):
    # Simula "Retry On Fail": max_tries intentos con una espera entre ellos (aquí solo se imprime, no se duerme).
    for intento in range(1, max_tries + 1):
        try:
            return {"ok": True, "intento": intento, "dato": fn()}
        except ConnectionError as e:
            print(f"  intento {intento}/{max_tries}: {e}", end="")
            if intento == max_tries:
                print("  → se agotaron los intentos: el error SUBE al workflow (y dispara el error workflow)")
                return {"ok": False, "error": str(e)}
            print(f"  → espera {espera_ms} ms")


print("Se recupera al 3.er intento:")
print(con_reintentos(api_inestable(3)))
print("\nNo se recupera:")
print(con_reintentos(api_inestable(99)))

**Qué observar.** Con 3 intentos, una falla transitoria se absorbe; una persistente **agota** los intentos y el error sube. En el laboratorio, el ticket `06_falla_cmdb` consulta un host cuya CMDB responde 503 a propósito: verás los intentos y, después, la entrada de `error` en la bitácora que escribe el *error workflow*.

## Sub-workflows

Un **sub-workflow** es un workflow que otro llama con el nodo **Execute Workflow** (o como herramienta con *Call n8n Workflow Tool*). Se usa para:

- **Reutilizar** (la consulta a la CMDB la pueden usar otros workflows).
- **Aislar y probar** una pieza por separado.
- **Mantener legibles** los lienzos grandes.

Cosas a tener en cuenta: el workflow llamado debe estar **publicado** para que el padre pueda publicarse (n8n lo exige al publicar), los datos entran por el disparador "cuando me llaman" (campos definidos o *passthrough*), y el resultado es la salida del **último nodo ejecutado**. Un error dentro del sub-workflow **sube** al padre.

---
# Parte 8 · Límites reales y operación

n8n resuelve mucho, pero **no es gratis de operar** ni sirve para todo. Estos son los límites que conviene conocer **antes** de apostar un proceso a él.

## Versionado y Git

- Un workflow es un **JSON**. Puedes exportarlo (editor, API pública o CLI: `n8n export:workflow`) y **guardarlo en Git** como cualquier archivo; así lo hace este curso (`ai-agents-playground/n8n-workflows/workflows`).
- Los *diffs* son ruidosos (posiciones de nodos, identificadores). Ayuda **nombrar bien los nodos**, mantener identificadores de workflow y de credencial **estables** y revisar los cambios en el editor, no solo en el texto.
- n8n incluye una función de **control de versiones con Git y entornos** (desarrollo, producción), pero la documentación indica que está disponible en los planes **Business y Enterprise**, no en la edición comunitaria. En la comunitaria, el flujo es el manual: exportar, versionar, importar por CLI (lo que hace `scripts/import_workflows.sh`).

## Pruebas

- Las pruebas nativas son **manuales**: ejecutar un paso, fijar datos (*pin data*), revisar la ejecución. n8n ha ido agregando **evaluaciones** para workflows con IA; revisa en la documentación qué incluye tu plan y tu versión.
- Lo que sí puedes automatizar tú: **validación estática** del JSON (la Parte 3), **pruebas contra los webhooks** con `pytest` y datos de ejemplo (el playground trae siete tickets, uno por ruta), y un **modo simulado** que evita depender de un modelo. El flujo de IA se evalúa como cualquier LLM: un conjunto fijo de casos con resultado esperado.
- Una regla del curso aplica aquí: **un conjunto fijo de casos que incluya los difíciles** (el ticket con inyección, el que falla, el inválido).

## Licencia: *fair-code*

- n8n usa la **Sustainable Use License** (y la *Enterprise License* para el código con `.ee.` en el nombre). Es **fair-code**: el código es visible y se puede usar, modificar y redistribuir, **con límites**; **no** es "open source" en el sentido de la OSI.
- Según la documentación oficial, la licencia permite usar o modificar el software **para tus propios fines de negocio internos o de uso personal/no comercial**; **distribuirlo o dárselo a otros** solo de forma gratuita y no comercial; y no se pueden quitar los avisos de licencia. Para ofrecerlo a terceros como producto o servicio **hace falta un acuerdo comercial**.
- Consecuencia práctica: automatizar los procesos de **tu empresa** con n8n auto-hospedado es el caso permitido; revender n8n, o embeberlo en un producto que ofreces a clientes, requiere **hablar con n8n**. Ante la duda, consulta las *License FAQs* o a tu área legal. (Esto es una guía, no asesoría legal.)

## Operación en *self-hosting*

| Tema | Qué considerar |
| --- | --- |
| **Base de datos** | Por defecto usa **SQLite** (suficiente para aprender); para producción, **PostgreSQL** |
| **Escalar** | El **modo cola** (*queue mode*) separa la instancia principal de **workers**; añade piezas que operar |
| **Versiones** | Sale una menor casi cada semana. **Fija la versión** de la imagen, lee las notas de versión y prueba antes de actualizar |
| **Copias de seguridad** | La base (workflows, credenciales cifradas, ejecuciones) **y la `N8N_ENCRYPTION_KEY`**, guardada **aparte**. Sin la clave, las credenciales respaldadas no se pueden leer |
| **Datos de ejecución** | Crecen sin parar: configura la **poda** (`EXECUTIONS_DATA_PRUNE`, `EXECUTIONS_DATA_MAX_AGE`) |
| **Monitoreo** | Endpoints de salud (`/healthz`, `/healthz/readiness`), logs y la lista de ejecuciones fallidas (más el *error workflow*) |
| **Disponibilidad** | Una sola instancia es un punto único de falla; los webhooks de producción dejan de responder si cae |
| **Esperas largas** | Un *Wait* de aprobación deja una ejecución **guardada en la base** hasta que alguien responde o caduca: planifica el volumen |

## Seguridad

| Riesgo | Defensa |
| --- | --- |
| **Credenciales** | Solo en el almacén de credenciales (cifrado con `N8N_ENCRYPTION_KEY`), nunca en nodos ni en el JSON; mínimo privilegio por credencial; rotarlas. Los workflows exportados llevan solo `id` y nombre |
| **Exposición de webhooks** | La URL **de producción** es pública si la instancia lo es. Publica solo lo necesario, detrás de **HTTPS** (proxy inverso), y no expongas el editor |
| **Autenticación de webhooks** | *Header Auth*, *Basic* o *JWT* en el nodo *Webhook* (el laboratorio usa cabecera `X-API-Key`); el nodo también admite lista de IP permitidas. Sin autenticación, cualquiera puede disparar el flujo |
| **Cifrado** | En reposo: credenciales cifradas con la clave de la instancia; en tránsito: TLS. Respalda la clave y considera rotarla (hay procedimiento documentado) |
| **Nodo Code y *Execute Command*** | Ejecutan código; limita quién edita workflows y bloquea nodos de riesgo; usa runners de tareas aislados. El comando `n8n audit` lista los nodos de riesgo |
| **Variables de entorno y red interna** | Evita leer secretos con `$env` desde nodos Code; hay opciones para bloquear ese acceso y para activar **protección SSRF** (qué hosts y rangos pueden alcanzar los nodos) |
| **Datos sensibles en ejecuciones** | Poda, acceso por rol y opción de ocultar datos de entrada y salida |
| **Inyección de instrucciones** | El texto que entra por un webhook y llega a un modelo es **dato no confiable**: lista blanca de acciones, aprobación humana, agente sin herramientas de escritura |
| **Copias de seguridad** | Probar la **restauración** (con la clave), no solo el respaldo |

> 💬 **Para pensar:** tu webhook de tickets está detrás de HTTPS y con token. Alguien roba el token. ¿Qué es lo **peor** que puede hacer con el flujo del curso? (Pista: mira qué puede disparar y qué necesita aprobación.)

---
# Parte 9 · (Opcional) Llamar al n8n local desde Python

Estas celdas **disparan los webhooks del n8n local** con `requests`: envían un ticket, leen la bitácora simulada y aprueban desde Python. Necesitan el n8n del playground levantado y los workflows importados (ver el README de `n8n-workflows`). En modo `simulado` **no cuestan nada**.

> ⏭️ Si no tienes n8n corriendo, déjalas desactivadas (`N8N_LOCAL = False` en la Parte 0): el resto del notebook no las necesita.

In [ ]:
# Opcional · necesita n8n en N8N_URL (por defecto http://localhost:5678) con los workflows importados.
import time

import requests


def token_webhook() -> str:
    # Lee N8N_WEBHOOK_TOKEN del entorno o del .env del playground. Nunca lo imprimas.
    if os.getenv("N8N_WEBHOOK_TOKEN"):
        return os.environ["N8N_WEBHOOK_TOKEN"]
    from dotenv import dotenv_values

    for base in [DIR_WORKFLOWS.parent if DIR_WORKFLOWS else None, Path(".")]:
        if base and (base / ".env").exists():
            valor = dotenv_values(base / ".env").get("N8N_WEBHOOK_TOKEN")
            if valor:
                return valor
    raise RuntimeError("Define N8N_WEBHOOK_TOKEN en el entorno o en el .env del playground")


def enviar_ticket(ticket: dict, api_key: str | None = None) -> requests.Response:
    cabeceras = {"X-API-Key": api_key if api_key is not None else token_webhook()}
    return requests.post(f"{N8N_URL}/webhook/tickets", json=ticket, headers=cabeceras, timeout=15)


def bitacora(tipo: str | None = None, ticket_id: str | None = None) -> list[dict]:
    params = {k: v for k, v in {"tipo": tipo, "ticket_id": ticket_id}.items() if v}
    return requests.get(f"{N8N_URL}/webhook/sim-registro", params=params, timeout=10).json()["entradas"]


if not N8N_LOCAL:
    print("⏭️ Omitida: pon N8N_LOCAL = True en la Parte 0 (necesita n8n corriendo).")
else:
    print("n8n:", requests.get(f"{N8N_URL}/healthz", timeout=5).json())
    ticket = {"ticket_id": "TCK-2001", "asunto": "VPN caída en la sucursal", "descripcion": "Nadie en la sucursal puede conectarse a la VPN desde las 7:40; el cliente dice gateway no responde.",
              "host": "vpn-gw-01", "solicitante": "usuario.ficticio99", "canal": "notebook"}
    r = enviar_ticket(ticket)
    print("Con la llave correcta →", r.status_code, r.json())
    print("Sin llave             →", requests.post(f"{N8N_URL}/webhook/tickets", json=ticket, timeout=15).status_code)

In [ ]:
# Opcional · espera la solicitud de aprobación y aprueba (o rechaza) desde Python.
DECISION = "Aprobar"        # ← "Aprobar" o "Rechazar"


def esperar(tipo, ticket_id, condicion=lambda e: True, segundos=30):
    limite = time.time() + segundos
    while time.time() < limite:
        for e in reversed(bitacora(tipo, ticket_id)):
            if condicion(e):
                return e
        time.sleep(1)


if N8N_LOCAL:
    aviso = esperar("notificacion", "TCK-2001", lambda e: e.get("url_aprobacion"))
    print("Aprobación pendiente:", aviso["mensaje"] if aviso else "no llegó")
    if aviso:
        # El formulario del nodo Wait recibe los campos en el orden definido: Decision, Aprobador, Comentario.
        campos = {"field-0": (None, DECISION), "field-1": (None, "OL"), "field-2": (None, "Desde el notebook")}
        print("Formulario →", requests.post(aviso["url_aprobacion"], files=campos, timeout=15).status_code)
        final = esperar("auditoria", "TCK-2001")
        print("Auditoría:", {k: final[k] for k in ("estado", "severidad", "dry_run") if k in final} if final else "no llegó")
else:
    print("⏭️ Omitida (N8N_LOCAL = False).")

> 💬 **Para pensar:** el cliente recibió `202` **antes** de que nadie aprobara nada. ¿Qué ventaja tiene para quien dispara el flujo? ¿Y qué desventaja (cómo se entera del resultado)? ¿Cómo lo resolverías (una URL de *callback*, consultar por `ticket_id`…)?

---
# Parte 10 · La demostración

La demostración sigue este orden (≈ 60 min con la práctica guiada). El detalle, con comandos, está en el README de `n8n-workflows`.

| Paso | Qué se hace | Concepto |
| --- | --- | --- |
| 1 | `docker compose up -d`, abrir `http://localhost:5678`, crear el propietario | *Self-hosting*, versión fijada, datos en un volumen |
| 2 | Crear las credenciales y **importar los workflows por CLI**; mirarlos en el editor | Credenciales, workflows como JSON |
| 3 | **Agente de chat** (workflow 01): preguntar "¿Cómo está la VPN?" y ver cómo usa las herramientas | AI Agent, memoria, herramientas, sub-workflow como herramienta |
| 4 | **Triaje** (workflow 02): enviar el ticket crítico con `curl`/Python; mostrar la ejecución en el editor | Webhook autenticado, expresiones, Switch, sub-workflow |
| 5 | Abrir el **formulario de aprobación**; aprobar; ver la acción en `dry_run` y la auditoría | Aprobación humana, `Wait`, `dry_run` |
| 6 | Enviar el ticket con **inyección**; ver que la propuesta no cambia | Datos no confiables, lista blanca |
| 7 | Enviar el ticket que **falla**; ver reintentos y el *error workflow* | Reintentos, error workflow |
| 8 | (Opcional) cambiar `modo` a `llm` con un modelo; (opcional) activar el nodo MCP | Modelos, MCP |
| 9 | `n8n audit` y repaso de la seguridad | Límites y operación |

**Práctica guiada.** Cada persona (a) importa los workflows, (b) dispara dos tickets distintos, (c) cambia **una cosa** del flujo (una palabra clave de la clasificación simulada, el tiempo de caducidad de la aprobación o la regla de prioridad) y comprueba el efecto, y (d) agrega una **nota** y un **nombre claro** a un nodo. Puesta en común: qué falló, qué sorprendió.

---
# Parte 11 · Cierre de la sesión y del Módulo 1

## Lo que aprendiste hoy

1. **n8n** es una herramienta de bajo código para **orquestar procesos e integrar sistemas**; el código propio gana donde hay **lógica compleja, pruebas y control fino**. Se combinan.
2. Un workflow son **triggers y nodos** que pasan **items**; las **expresiones** leen datos; las **credenciales** viven cifradas fuera del JSON; cada **ejecución** se guarda y se puede depurar.
3. El **AI Agent** es el bucle de herramientas de las sesiones 8 y 9 hecho con conexiones: modelo, memoria, herramientas y parser de salida. La **salida estructurada** garantiza la forma; **la verdad se valida en código**.
4. Un **sub-workflow** puede ser una herramienta del agente, y n8n puede ser **cliente y servidor MCP**.
5. El patrón seguro: el agente **propone**, una **persona aprueba** y el **flujo ejecuta** (en `dry_run`, con lista blanca, caducidad y auditoría).
6. **Errores:** reintentos para lo transitorio, *error workflow* para avisar, caducidad para esperas. Los reintentos de escrituras exigen idempotencia.
7. **Límites reales:** Git y entornos integrados solo en planes de pago, pruebas sobre todo manuales, licencia *fair-code*, y una operación en *self-hosting* que incluye actualizaciones, copias de seguridad y la clave de cifrado.
8. **Seguridad:** webhooks con autenticación y HTTPS, credenciales cifradas, mínimo privilegio, sin secretos en nodos, y el texto que llega al modelo es **dato no confiable**.

## 🏁 Repaso del Módulo 1 (S1 a S12)

El hilo del módulo fue **LLM → RAG → agente con herramientas (MCP) → automatización**.

| Bloque | Sesiones | Lo que te llevas |
| --- | --- | --- |
| **El LLM y su ecosistema** | S1 a S3 | Tokens, ventana de contexto y costo; consumo vía API (streaming, errores, salida estructurada); prompting y *context engineering* |
| **Embeddings y RAG** | S4 a S7 | Embeddings y bases vectoriales (Neon, Pinecone); un RAG simple con citas; chunking, búsqueda híbrida y reranking; RAG avanzado, GraphRAG y evaluación |
| **Agentes** | S8 y S9 | Agentes con LangChain (herramientas, bucle, traza); salida estructurada, middleware, memoria, contexto de ejecución y RAG agéntico |
| **MCP** | S10 | Arquitectura, seguridad y un servidor propio con lecturas **y escrituras** protegidas |
| **LangGraph** | S11 | Un agente con RAG, memoria, herramientas y MCP, con aprobación humana antes de escribir |
| **n8n** | S12 | Orquestar procesos de TI con workflows, agentes y aprobaciones |

**Cuatro ideas que atraviesan todo el módulo:**

1. **El modelo propone; tu código dispone.** Herramientas, validaciones, listas blancas, aprobación humana.
2. **La forma no es la verdad.** Esquemas, citas y clasificaciones se verifican en código y se **miden** con un conjunto fijo de casos.
3. **Empieza por lo más simple que funcione** (un prompt, un RAG fijo, un workflow) y agrega autonomía donde una medición lo justifique.
4. **Costo, latencia y seguridad son requisitos**, no extras: cada vuelta de un agente son llamadas, y cada texto externo es una posible instrucción maliciosa.

## ➡️ Qué viene: Módulo 2

El **Módulo 2** es **AI Engineer II: Arquitectura y Seguridad de Sistemas Agénticos** (34 h, 17 sesiones, 6 semanas). Lleva el agente del Módulo 1 a **un sistema multi-agente seguro**. Según el programa del curso, cubre:

- **Arquitectura multi-agente:** patrones (orquestador-trabajador, jerárquico, *peer-to-peer*), cuándo conviene más de un agente y las capas del *harness* (herramientas, contexto, memoria, bucles de control, permisos, evaluaciones y observabilidad).
- **Frameworks de orquestación** y **comunicación entre agentes** con el protocolo **A2A**.
- **Orquestación avanzada y memoria:** grafos de estado, memoria compartida, *checkpointing* y ejecución durable.
- **Guardrails** de entrada, salida y comportamiento; OWASP para aplicaciones LLM y agénticas.
- **Seguridad de agentes:** *prompt injection*, riesgos de MCP y A2A, identidad y autorización.
- **Control humano, *red teaming* y evaluación y testing de agentes**; resiliencia y gobernanza.

> Es la continuación natural de hoy: lo que en esta clase resolvimos con **una lista blanca, una aprobación y un `dry_run`** se estudia allí de forma sistemática (guardrails, identidad, *human-in-the-loop*, *red teaming*). El agente final del Módulo 1 es el punto de partida.

## ✅ Autoevaluación

1. Menciona tres criterios para decidir entre un workflow de n8n y código propio, y una situación donde **combinarlos** sea lo mejor.
2. ¿Qué es un **item** en n8n y qué diferencia hay entre `{{ $json.campo }}` y `{{ $('Nodo').first().json.campo }}`?
3. ¿Por qué un workflow exportado **no** contiene secretos, y qué pasa con las credenciales si pierdes la `N8N_ENCRYPTION_KEY`?
4. ¿Qué cuatro tomas tiene un nodo **AI Agent** y a qué concepto de LangChain equivale cada una?
5. ¿Por qué el nombre del nodo de una herramienta importa para el modelo?
6. En el triaje, ¿por qué el agente **no** tiene herramientas de escritura y qué pasa entre la propuesta y la acción?
7. Diferencia entre **reintento del nodo**, ***error workflow*** y **caducidad del *Wait***. ¿Por qué reintentar una escritura es peligroso?
8. ¿Qué partes de n8n puedes tener en la edición comunitaria y cuáles, según la documentación, requieren un plan de pago (pista: versionado con Git)?
9. ¿Qué permite y qué no permite la **Sustainable Use License**, dicho con tus palabras?
10. Tu instancia expone un webhook a Internet. Nombra cuatro medidas de seguridad.
11. ¿Qué papel cumplen el **servidor** y el **cliente MCP** de n8n y qué riesgo agregas al exponer un servidor?
12. De las cuatro ideas que atraviesan el módulo, ¿cuál te cambió más la forma de pensar un sistema con IA?

## 🚀 Retos

- En el nodo **Config** del triaje, cambia `dry_run` a `false` y observa el estado final (`ejecutada` en vez de `ejecutada_dry_run`). ¿Por qué `true` es el valor por defecto?
- Agrega una **tercera categoría de acción** (por ejemplo `abrir_ticket_proveedor`): ¿en cuántos sitios debes tocar la lista blanca (esquema y código)? El test `test_lista_blanca_de_acciones_coincide_en_esquema_y_codigo` te avisa si olvidas uno.
- Añade al revisor de la Parte 3 una regla: "todo *Wait* debe tener `limitWaitTime`".
- Escribe un **cuarto ticket** de ejemplo (ficticio) que provoque una clasificación `critica` con otro host de la CMDB simulada.
- Conecta el **MCP Client Tool** del workflow 01 al servidor del workflow 07 (o al de la Sesión 10) y observa qué herramientas aparecen.
- Sustituye la bitácora simulada por una **base de datos real** (por ejemplo, una tabla en Postgres) y decide qué credencial necesita y con qué permisos mínimos.

## 📖 Glosario

| Término | Definición |
| --- | --- |
| **n8n** | Herramienta de automatización de flujos con editor visual, auto-hospedable, con nodos de IA |
| **Workflow** | Conjunto de nodos conectados, guardado como JSON |
| **Trigger** | Nodo que inicia un workflow (webhook, chat, programación, error, MCP, "cuando me llaman") |
| **Nodo** | Un paso de un workflow: acción, lógica o IA |
| **Item** | Unidad de datos que viaja entre nodos (`{"json": {...}}`) |
| **Expresión** | Valor calculado con `{{ }}` (JavaScript) a partir de datos de nodos y de la ejecución |
| **Credencial** | Secreto cifrado guardado en n8n y referenciado por nombre desde los nodos |
| **Ejecución** | Una corrida de un workflow con los datos de cada nodo, guardada para depurar |
| **Pin data** | Fijar la salida de un nodo para iterar sin volver a dispararlo |
| **AI Agent (nodo)** | Agente con bucle de herramientas configurado con conexiones (modelo, memoria, herramientas, parser) |
| **Information Extractor** | Nodo que extrae campos de un texto con un esquema |
| **Sub-workflow** | Workflow llamado por otro (Execute Workflow) o usado como herramienta |
| **MCP Client Tool / MCP Server Trigger** | Nodos que hacen de cliente o servidor del Model Context Protocol |
| **Wait** | Nodo que pausa la ejecución hasta un tiempo, un webhook o un formulario (con caducidad opcional) |
| **Error workflow** | Workflow con *Error Trigger* que corre cuando otro workflow falla |
| **Retry On Fail** | Ajuste de nodo que reintenta ante una falla |
| **`dry_run`** | Modo en que la acción se registra pero no se ejecuta de verdad |
| **Lista blanca** | Conjunto de acciones permitidas, verificado en código |
| **Fair-code** | Modelo de licencia con código visible y usos limitados (no es open source de la OSI) |
| **Queue mode** | Modo de escala con instancia principal y *workers* |
| **`N8N_ENCRYPTION_KEY`** | Clave que cifra las credenciales guardadas; hay que respaldarla |
| **Inyección indirecta** | Instrucciones maliciosas escondidas en datos que el modelo lee |
| **Módulo 2** | AI Engineer II: arquitectura y seguridad de sistemas agénticos |

## ➡️ Siguiente

Fin del **Módulo 1**. Sigue el **Módulo 2**: de un agente a un sistema multi-agente seguro.